## Physics-Motivated Geometric Representation for Propagation-State Prediction in 3D UAV Communication Environments

## Part 1: Physics-motivated Geometric Propagation Design

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ezdxf
from collections import Counter
from pathlib import Path
import os
from matplotlib.patches import Polygon


## 2. Experiment Configuration
This section defines the main parameters used for the experiments.
    

In [2]:
BASE_DIR = Path(".")
REPRESENTATION_DIR = BASE_DIR / "Inputs_DIR"
OUTPUT_DIR = BASE_DIR / "REPRESENTATION_DIR"
EPS = 0.1

In [3]:
features1 = [
    ####### 3D Link Geometry Features  ###############
    "distance_3d",   
    "elevation_angle_deg",   
    "tx_height",   
    "rx_height",    
    ###### LOS Blockage Geometry Features  ###############
    "blocking_Candidate_number", 
    "blocking_score", 
    "blocking_position", 
    "blocking_alignment", 
    "blocking_margin", 
    "blocking_Delta_height", 
    ######################  Geometric Multipath Environment Features  ###########    
    "blocker_count",   
    "sum_blocker", 
    "occupancy",  
    "blocking_potential", 
    "reflection_potential",  
    "effective_reflect_area", 
    "log_reflection_ratio", 
    #################   Local Spatial Confinement Features  ############    
    "local_confinement", 
    "normalize_local_confinement", 
]


##  Dataset-specific input/output configuration.

In [4]:


cities = {
    1: {
        "name": "Seoul",
        "uav_file" : REPRESENTATION_DIR / "Seoul_South Korea_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR / "Seoul_South Korea_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR / "Seoul_Buildings.dxf",
        "output_file" : OUTPUT_DIR / "Seoul_Our_features.csv",        
    },
    2: {
        "name": "Boston",
        "uav_file" : REPRESENTATION_DIR / "BOSTON_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR / "BOSTON_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR / "Boston_Building.dxf",
        "output_file" : OUTPUT_DIR / "Boston_Our_features.csv",        
    },
    3: {
        "name": "Pisa_East",
        "uav_file" : REPRESENTATION_DIR / "Pisa_East_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR / "Pisa_East_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR / "Pisa_East_Building.dxf",
        "output_file" : OUTPUT_DIR / "Pisa_East_Our_features.csv",        
    },
    4: {
        "name": "Pisa_West",
        "uav_file": REPRESENTATION_DIR / "Pisa_West_UAV_Cartesian.csv",
        "gnd_file": REPRESENTATION_DIR / "Pisa_West_GND_Cartesian.csv",
        "dxf_file": REPRESENTATION_DIR / "Pisa_West_Building.dxf",
        "output_file" : OUTPUT_DIR / "Pisa_West_Our_features.csv",        
    },
    5: {
        "name": "London",
        "uav_file": REPRESENTATION_DIR / "London_UAV_Cartesian.csv",
        "gnd_file": REPRESENTATION_DIR / "London_GND_Cartesian.csv",
        "dxf_file": REPRESENTATION_DIR / "London_Building.dxf",
        "output_file" : OUTPUT_DIR / "London_Our_features.csv" ,        
    },
    6: {
        "name": "Manhattan",
        "uav_file" : REPRESENTATION_DIR /  "Manhattan_UAV_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR /  "Manhattan_GND_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR /  "Manhattan_Building.dxf",
        "output_file"  : OUTPUT_DIR /  "Manhattan_Our_features.csv" ,        
    },
    7: {
        "name": "Mobile_Seoul1",
        "uav_file" : REPRESENTATION_DIR /  "Mobile_Seoul_South Korea_UAV_TR_Cartesian.csv",
        "gnd_file" : REPRESENTATION_DIR /  "Mobile_Seoul_South Korea_GND_TR_Cartesian.csv",
        "dxf_file" : REPRESENTATION_DIR /  "Seoul_Buildings.dxf",
        "output_file" : OUTPUT_DIR /  "Mobile_Seoul_South Korea_TR_Our_features.csv" ,        
    },   
}


    

## Experiment Functions
The following functions implement data loading, and feature extraction used by the experiments.


 ## 1. IO-Related Functions
 -read_tx_rx
 -extract_faces_from_dxf

In [ ]:
def read_tx_rx(uav_file, gnd_file):
    """
    Load transmitter and receiver coordinates from CSV files.

    Parameters
    ----------
    uav_file : path-like
        CSV file containing UAV/transmitter coordinates.
    gnd_file : path-like
        CSV file containing ground/receiver coordinates.

    Returns
    -------
    uav_df : pandas.DataFrame
        Original UAV data.
    gnd_df : pandas.DataFrame
        Original ground-receiver data.
    uav_coords : numpy.ndarray, shape (N_tx, 3)
        Cartesian transmitter coordinates.
    gnd_coords : numpy.ndarray, shape (N_rx, 3)
        Cartesian receiver coordinates.

    Notes
    -----
    The CSV files are expected to contain the columns
    ``Cartesian_X``, ``Cartesian_Y``, and ``Cartesian_Z``.
    """
    uav_df = pd.read_csv(uav_file)
    gnd_df = pd.read_csv(gnd_file)    
    uav_coords = uav_df[['Cartesian_X', 'Cartesian_Y', 'Cartesian_Z']].values
    gnd_coords = gnd_df[['Cartesian_X', 'Cartesian_Y', 'Cartesian_Z']].values    
    return uav_df, gnd_df, uav_coords, gnd_coords


def extract_faces_from_dxf(dxf_file):
    """
    Extract valid vertical wall faces from a DXF model.

    Parameters
    ----------
    dxf_file : path-like
        DXF file containing building geometry represented by 3DFACE entities.

    Returns
    -------
    list of numpy.ndarray
        List of wall faces. Each face is represented by an array of
        3D Cartesian vertices with shape (N, 3).

    Notes
    -----
    Only DXF ``3DFACE`` entities classified as vertical wall faces by
    ``is_wall_face`` are retained. Degenerate faces and duplicated closing
    vertices are removed.
    """
    doc = ezdxf.readfile(dxf_file)
    msp = doc.modelspace()
    wall_faces = []
    for entity in msp.query("3DFACE"):
        vertices = []
        for vertex in [entity.dxf.vtx0,entity.dxf.vtx1,entity.dxf.vtx2,entity.dxf.vtx3]:
            if vertex is not None:
                vertices.append([float(vertex[0]),float(vertex[1]),float(vertex[2])])
        if len(vertices) < 3:
            continue
        face = np.array(vertices)
        # DXF 3DFACE may repeat the first vertex as the fourth vertex.
        # Remove duplicated last vertex
        if len(face) == 4:
            if np.linalg.norm(face[3] - face[0]) < 1e-9:
                face = face[:3]
        if is_wall_face(face):
            wall_faces.append(face)
    return wall_faces
    

## 2. Geometry-Related Functions
 -Compute_face_normal
 -Compute_face_area
 -Is_wall_face
 -Line_intersects_face
 -Los_local_coordinates

In [ ]:
def compute_face_normal(face):
    """
    Compute a unit normal vector for a polygonal face.

    Parameters
    ----------
    face : ndarray, shape (N, 3)
        Cartesian coordinates of the face vertices.

    Returns
    -------
    ndarray, shape (3,)
        Unit normal vector. Returns the zero vector for degenerate faces.

    Notes
    -----
    The normal is computed from the cross product of the first two
    polygon edges. The orientation of the returned normal depends on
    the vertex ordering.
    """
    face = np.asarray(face)
    if face.shape[0] < 3:
        return np.array([0.0, 0.0, 0.0])
    v1 = face[1] - face[0]
    v2 = face[2] - face[0]
    normal = np.cross(v1, v2)
    norm = np.linalg.norm(normal)
    if norm < 1e-12:
        return np.array([0.0, 0.0, 0.0])
    return normal / norm

def compute_face_area(face):
    """
    Compute area of a polygonal face.

    Parameters
    ----------
    face : ndarray
        Vertices of face, shape (N,3)

    Returns
    -------
    area : float
        Surface area of the face
    """
    # The polygon is triangulated using the first vertex as a common
    # reference point.
    # Triangulate the polygon using a fan from the first vertex.
    face = np.asarray(face)
    if len(face) < 3:
        return 0.0
    area = 0.0
    p0 = face[0]    
    for i in range(1, len(face)-1):
        v1 = face[i] - p0
        v2 = face[i+1] - p0
        triangle_area = 0.5 * np.linalg.norm(np.cross(v1, v2))
        area += triangle_area
    return area
    
def is_wall_face(face, nz_threshold=0.3):
    """
    Determine whether a face is approximately vertical.

    Parameters
    ----------
    face : ndarray, shape (N, 3)
        Polygon vertices.
    nz_threshold : float, default=0.3
        Maximum allowed absolute z-component of the unit surface normal.

    Returns
    -------
    bool
        True if the face is classified as a vertical wall.

    Notes
    -----
    A vertical wall has a surface normal primarily in the horizontal plane,
    resulting in a small absolute z-component.
    """
    normal = compute_face_normal(face)
    if np.linalg.norm(normal) < 1e-12:
        return False
    nz = abs(normal[2])
    return nz < nz_threshold

def line_intersects_face(p0, p1, face, tol=1e-8):
    """
    Check if a 3D line segment intersects a planar polygon face.
    Parameters
    ----------
    p0 : array-like
        Line start point.
    p1 : array-like
        Line end point.
    face : array-like
        Polygon vertices (N,3).
    tol : float
        Numerical tolerance.
    Returns
    -------
    bool
        True if the line segment intersects the face.
    """
     
    # The segment is first intersected with the plane containing the polygon.
    # The resulting 3D intersection point is then projected to a 2D plane
    # by dropping the dominant component of the face normal, after which
    # a point-in-polygon test is applied.

    p0 = np.asarray(p0, dtype=float)
    p1 = np.asarray(p1, dtype=float)
    face = np.asarray(face, dtype=float)
    if len(face) < 3:
        return False
    # Compute the face plane.
    v1 = face[1] - face[0]
    v2 = face[2] - face[0]
    normal = np.cross(v1, v2)
    norm = np.linalg.norm(normal)
    if norm < tol:
        return False
    normal = normal / norm
    # Solve p = p0 + t*d for the line-plane intersection parameter.
    d = p1 - p0
    denom = np.dot(normal, d)    
    if abs(denom) < tol:
        return False
    t = np.dot(normal, face[0] - p0) / denom
    # Intersection outside segment
    if t < -tol or t > 1 + tol:
        return False
    intersection = p0 + t * d
   # Project to 2D for the polygon containment test.
    abs_normal = np.abs(normal)
    if abs_normal[0] >= abs_normal[1] and abs_normal[0] >= abs_normal[2]:
        # drop x
        poly2d = face[:, [1, 2]]
        point2d = intersection[[1, 2]]
    elif abs_normal[1] >= abs_normal[0] and abs_normal[1] >= abs_normal[2]:
        # drop y
        poly2d = face[:, [0, 2]]
        point2d = intersection[[0, 2]]
    else:
        # drop z
        poly2d = face[:, [0, 1]]
        point2d = intersection[[0, 1]]
    # Apply a 2D ray-crossing point-in-polygon test.
    inside = False
    n = len(poly2d)
    j = n - 1
    for i in range(n):
        xi, yi = poly2d[i]
        xj, yj = poly2d[j]
        intersect = (((yi > point2d[1]) != (yj > point2d[1])) and      
            ( point2d[0] < (xj - xi)* (point2d[1] - yi)/(yj - yi + tol) + xi))
        if intersect:
            inside = not inside
        j = i
    return inside

def los_local_coordinates(tx, rx):
    """
    Construct a receiver-oriented local coordinate system aligned with the LOS.

    Parameters
    ----------
    tx : array-like, shape (3,)
        Transmitter Cartesian coordinates.
    rx : array-like, shape (3,)
        Receiver Cartesian coordinates.

    Returns
    -------
    R : numpy.ndarray, shape (3, 3)
        Rotation matrix whose rows are the local x-, y-, and z-axis unit
        vectors expressed in the global coordinate system.

    Notes
    -----
    The local x-axis is aligned with the transmitter-to-receiver LOS
    direction. The local z-axis is obtained by projecting the global
    vertical direction onto the plane perpendicular to the LOS. For a
    nearly vertical LOS, an alternative reference axis is used to avoid
    numerical degeneracy. The local y-axis completes a right-handed
    orthonormal frame.
    """
    tx = np.asarray(tx)
    rx = np.asarray(rx)
    los = rx - tx
    norm = np.linalg.norm(los)
    if norm < 1e-12:
        raise ValueError("TX and RX cannot have identical coordinates.")
    x_axis = los / norm
    # Start from global vertical
    z_global = np.array([0.0, 0.0, 1.0])
    # Remove LOS component
    # Project the global vertical direction onto the plane normal to the LOS.
    z_axis = (z_global - np.dot(z_global, x_axis) * x_axis)
    z_norm = np.linalg.norm(z_axis)
    # Nearly vertical LOS case
    if z_norm < 1e-12:
        if abs(x_axis[0]) < 0.9:
            temp = np.array([1.0,0.0,0.0])
        else:
            temp = np.array([0.0,1.0,0.0])
        z_axis = (temp - np.dot(temp,x_axis)*x_axis )
        z_axis /= np.linalg.norm(z_axis)
    else:
        z_axis /= z_norm
    y_axis = np.cross(z_axis, x_axis)
    y_axis /= np.linalg.norm(y_axis)
    R = np.vstack([x_axis,y_axis,z_axis])
    return R




## 3. Features-Related Functions
 -filter_faces_along_los
 -compute_representative_blocker
 -compute_reflector_blocker_features
 -compute_visible_blocking_features
 -compute_local_confinement_index
 -empty_features

In [5]:


def filter_faces_along_los(faces_local, x_min=0, x_max=100, radial_bound=50):
    """
    Select candidate surfaces near the LOS and identify exact LOS intersections.

    Parameters
    ----------
    faces_local : list of numpy.ndarray
        Surface faces represented in the LOS-oriented coordinate system.
    x_min : float, default=0
        Minimum allowed position along the local LOS x-axis.
    x_max : float
        LOS distance between transmitter and receiver.
    radial_bound : float, default=50
        Maximum perpendicular distance from the LOS axis for candidate
        surfaces.

    Returns
    -------
    filtered : list
        Faces whose centroids fall within the candidate LOS corridor.
    indices : list
        Original indices of the filtered faces.
    los_intersect : list
        Faces that geometrically intersect the LOS segment.
    los_indices : list
        Original indices of the intersecting faces.

    Notes
    -----
    The centroid-based filter is used as a computationally inexpensive
    candidate-selection step. Exact LOS intersection is subsequently
    determined using ``line_intersects_face``.
    """
    # Extend the candidate corridor beyond the receiver to retain nearby
    # surfaces whose centroids fall slightly beyond the LOS endpoint.
    extension = max(20.0, 0.2 * x_max)
    x_limit = x_max + extension
    filtered = []
    indices = []
    los_intersect = []
    los_indices = []
    p0 = np.array([0.0, 0.0, 0.0])
    p1 = np.array([x_max , 0.0, 0.0])
    for idx, face in enumerate(faces_local):
        face = np.asarray(face)
        centroid = np.mean(face, axis=0)
        x = centroid[0]
        radial_distance = np.linalg.norm(centroid[1:])
        # Existing centroid-based filter
        # Candidate surfaces: inexpensive centroid-based LOS corridor filter.
        if (x_min <= x <= x_limit) and (radial_distance <= radial_bound):
            filtered.append(face)
            indices.append(idx)
        # Exact LOS intersection test
        # Identify faces that physically intersect the TX-RX LOS segment.
        # Exact LOS blockers: surfaces whose polygon intersects the finite TX-RX segment.
        if line_intersects_face(p0, p1, face):
            los_intersect.append(face)
            los_indices.append(idx)
    return filtered, indices, los_intersect, los_indices

def compute_representative_blocker(faces_global, faces_local, tx, rx):
    """
    Compute the dominant LOS-blocking surface using a geometry-based score.

    Parameters
    ----------
    faces_global : list of numpy.ndarray
        Candidate LOS-intersecting faces in global coordinates.
    faces_local : list of numpy.ndarray
        Corresponding candidate faces in the LOS-oriented coordinate system.
    tx : array-like, shape (3,)
        Transmitter Cartesian coordinates.
    rx : array-like, shape (3,)
        Receiver Cartesian coordinates.

    Returns
    -------
    dict
        Dictionary containing:
        
        ``blocking_Candidate_number``
            Number of valid candidate blocking faces.
        ``blocking_score``
            Maximum geometric blocking score.
        ``blocking_position``
            Normalized position of the selected blocker along the LOS.
        ``blocking_alignment``
            Absolute alignment between the face normal and LOS direction.
        ``blocking_margin``
            Perpendicular distance from the highest face point to the LOS.
        ``blocking_Delta_height``
            Vertical clearance between the LOS and highest face point.

    Notes
    -----
    The blocking score combines surface orientation, lateral proximity to
    the LOS, and vertical clearance. The face with the largest score is
    selected as the representative blocker.
    """
    empty_result = {
        "blocking_Candidate_number": np.nan,
        "blocking_score": np.nan,
        "blocking_position": np.nan,
        "blocking_alignment": np.nan,
        "blocking_margin": np.nan,
        "blocking_Delta_height": np.nan
    }
    if len(faces_global) == 0:
        return empty_result
    tx = np.asarray(tx)
    rx = np.asarray(rx)
    los = rx - tx
    distance = np.linalg.norm(los)
    if distance < 1e-12:
        return empty_result
    los_unit = los / distance
    scores = []
    for fg, fl in zip(faces_global, faces_local):
        fg = np.asarray(fg)
        fl = np.asarray(fl)
        normal = compute_face_normal(fl)
        if np.linalg.norm(normal) < 1e-12:
            continue
        # Measure how strongly the surface normal is aligned with the LOS direction.
        alignment = abs(np.dot(normal, np.array([1, 0, 0])))
        centroid_local = np.mean(fl, axis=0)
        position = centroid_local[0] / distance
        # Use the highest face vertex to characterize the potential vertical extent
        # of the surface relative to the LOS.
        p = fg[np.argmax(fg[:, 2])]
        # Projection of point onto LOS
        # Project the highest surface point onto the finite TX-RX LOS segment.
        projection = np.dot(p - tx, los_unit)
        projection = np.clip(projection, 0, distance)
        foot = tx + projection * los_unit
        # 3D distance from point to LOS
        margin = np.linalg.norm(p - foot)        
        # Vertical clearance
        # Positive means the point is below the LOS
        delta_h = foot[2] - p[2]
        # Penalize surfaces that remain below the LOS path.
        clearance_penalty = 1.0 / (1.0 + max(delta_h, 0))
        # Combine LOS alignment, lateral proximity, and vertical clearance.
        # Small numerical regularization prevents excessive scores when the
        # surface approaches the LOS.
        score = (alignment**2 / (margin + EPS)) * clearance_penalty
        scores.append({
                "score": score,
                "position": position,
                "alignment": alignment,
                "margin": margin,
                "height": delta_h
            })
    if len(scores) == 0:
        return empty_result
    best = max(scores, key=lambda x: x["score"])
    return {
        "blocking_Candidate_number": len(scores),
        "blocking_score": best["score"],
        "blocking_position": best["position"],
        "blocking_alignment": best["alignment"],
        "blocking_margin": best["margin"],
        "blocking_Delta_height": best["height"]
    }
    
def compute_visible_blocking_features(faces_global, tx,rx,az_bins=340,el_bins=36,score_threshold=0.65):
    
    if len(faces_global) == 0:
         return 0, 0, 0, 0
    tx=np.asarray(tx)
    rx=np.asarray(rx)
    los = rx-tx
    los_norm=np.linalg.norm(los)
    if los_norm < 1e-12:
         return 0, 0, 0, 0
    los_unit = los/los_norm
    # ------------------------------------------------
    # Keep nearest face in each viewing direction
    # ------------------------------------------------
    visible_faces = {}
    for face in faces_global:
        face=np.asarray(face)    
        centroid=np.mean(face,axis=0)    
        vec=centroid-rx
        dist=np.linalg.norm(vec)    
        if dist < 1e-12:
            continue    
        az=np.arctan2(vec[1],vec[0])    
        az_id=int((az+np.pi)/(2*np.pi)*az_bins)    
        az_id=np.clip(az_id,0,az_bins-1)    
        if az_id not in visible_faces or dist < visible_faces[az_id]["distance"]:
            visible_faces[az_id]={
                "face":face,
                "distance":dist,
                "centroid":centroid
            }    
    blocker_count=0
    sum_blocker = 0
    face_count = len(visible_faces)
    for item in visible_faces.values():
        face=item["face"]
        centroid=item["centroid"]
        normal=compute_face_normal(face )
        if np.linalg.norm(normal)<1e-12:
            continue        
        face_dir = centroid-rx
        face_dir /= np.linalg.norm(face_dir)        
        d_tx = centroid - tx
        d_tx /= np.linalg.norm(d_tx)
        if np.dot(normal, d_tx) > 0:
            normal = -normal
        # direction from face to RX
        d_rx = rx - centroid
        d_rx /= np.linalg.norm(d_rx)       
        block_phi = np.arccos(np.clip(np.dot(-face_dir, los_unit), -1, 1))        
        blocking_score = (abs(np.dot(normal, los_unit))*np.cos(block_phi/2))
        sum_blocker = sum_blocker + blocking_score        
        if blocking_score >= score_threshold:
            blocker_count += 1
    return  blocker_count/face_count, sum_blocker

def compute_reflector_blocker_features(faces_global,tx,rx,az_bins=72,el_bins=18,distance_decay=25.0):    
    """
    Compute receiver-centered geometric descriptors for surrounding surfaces.

    Parameters
    ----------
    faces_global : list of numpy.ndarray
        Building wall faces in global Cartesian coordinates.
    tx : array-like, shape (3,)
        Transmitter Cartesian coordinates.
    rx : array-like, shape (3,)
        Receiver Cartesian coordinates.
    az_bins : int, default=72
        Number of azimuth bins used for angular visibility discretization.
    el_bins : int, default=18
        Number of elevation bins used for angular visibility discretization.
    distance_decay : float, default=25.0
        Distance-decay scale used in the geometric weighting.

    Returns
    -------
    dict
        Geometric descriptors containing angular occupancy, reflection
        potential, blocking potential, effective reflection area, and the
        logarithmic reflection-to-blocking ratio.

    Notes
    -----
    For each angular bin, only the nearest surface centroid is retained
    as an approximation of receiver-side visibility. Surface contributions
    are weighted according to area, distance, orientation, and angular
    position relative to the LOS.
    
    The resulting quantities are geometric proxies rather than
    electromagnetic reflection or diffraction coefficients.
    """
    if len(faces_global) == 0:
        return empty_features()
    tx = np.asarray(tx, dtype=float)
    rx = np.asarray(rx, dtype=float)
    los = rx - tx
    los_norm = np.linalg.norm(los)
    if los_norm < 1e-12:
        return empty_features()
    los_unit = los / los_norm
    # Retain only the nearest surface in each angular direction as a
    # low-cost approximation of receiver-side visibility.
    visible = {}
    for face in faces_global:
        face = np.asarray(face)
        centroid = face.mean(axis=0)
        vec = centroid - rx
        dist = np.linalg.norm(vec)
        if dist < 1e-8:
            continue
        direction = vec / dist
        az = np.arctan2(direction[1], direction[0])
        el = np.arcsin(direction[2])
        az_bin = int((az + np.pi) / (2*np.pi) * az_bins)
        el_bin = int((el + np.pi/2) / np.pi * el_bins)
        az_bin = np.clip(az_bin, 0, az_bins-1)
        el_bin = np.clip(el_bin, 0, el_bins-1)
        key = (az_bin, el_bin)
        if key not in visible or dist < visible[key]["distance"]:
            visible[key] = {
                "face": face,
                "centroid": centroid,
                "distance": dist
            }
    ############################################################
    reflection_sum = 0.0
    blocking_sum = 0.0
    reflect_area = 0.0
    for item in visible.values():
        face = item["face"]
        centroid = item["centroid"]
        dist = item["distance"]
        normal = compute_face_normal(face)
        n_norm = np.linalg.norm(normal)
        if n_norm < 1e-12:
            continue
        normal /= n_norm
        # Compute surface area for area-weighted geometric contributions.
        area = compute_face_area(face)
        # Orient the normal toward the transmitter-facing side of the surface.
        d_tx = centroid - tx
        d_tx /= np.linalg.norm(d_tx)
        d_rx = rx - centroid
        d_rx /= np.linalg.norm(d_rx)        
        if np.dot(normal, d_tx) > 0:
            normal = -normal
        # Compare the actual surface normal with the ideal specular-reflection
        # normal defined by the TX-to-surface and surface-to-RX directions.
        specular_normal = d_rx - d_tx
        specular_normal /= np.linalg.norm(specular_normal)
        reflection_quality = abs(np.dot(normal, specular_normal))
        # Weight surfaces according to their angular position relative to the LOS.
        # Apply exponential distance attenuation to emphasize nearby structures.
        # Accumulate reflection and blocking geometric potentials.
        face_dir = centroid - rx
        face_dir /= np.linalg.norm(face_dir)
        phi = np.arccos(np.clip(np.dot(face_dir, los_unit), -1.0, 1.0))                  
        block_weight = np.cos(phi/2.0)**2
        reflect_weight = np.sin(phi/2.0)**2        
        dist_weight = np.exp(-dist / distance_decay)        
        reflection_score = ( area * dist_weight * reflection_quality * reflect_weight )        
        blocking_score = ( area * dist_weight * abs(np.dot(normal, los_unit)) * block_weight)
        reflection_sum += reflection_score
        blocking_sum += blocking_score
        reflect_area += area * reflection_quality * dist_weight
    occupancy = len(visible) / float(az_bins * el_bins)
    log_reflection_ratio = np.log((reflection_sum+1e-8) /(blocking_sum+1e-8))    
    return {
        "occupancy": occupancy,
        "reflection_potential": reflection_sum,
        "blocking_potential": blocking_sum,
        "effective_reflect_area": reflect_area,        
        "log_reflection_ratio":log_reflection_ratio
    }

def empty_features():
    """
    Return zero-valued geometric descriptors for environments with no
    valid visible surfaces.
    """
    return {
        "occupancy": 0.0,
        "reflection_potential": 0.0,
        "blocking_potential": 0.0,
        "effective_reflect_area": 0.0,        
        "log_reflection_ratio": 0.0,
    }
    
def compute_local_confinement_index(faces_global,rx,score_threshold=1.0,normalize=True):    
    """
    Compute the local spatial confinement index around the receiver.

    Parameters
    ----------
    faces_global : list of numpy.ndarray
        Building wall faces in global Cartesian coordinates.
    rx : array-like, shape (3,)
        Receiver Cartesian coordinates.
    score_threshold : float, default=1.0
        Minimum effective-height-to-distance ratio required for a surface
        to contribute to the confinement estimate.
    normalize : bool, default=True
        If True, return the mean contribution over valid candidate surfaces.
        If False, return the accumulated contribution.

    Returns
    -------
    confinement : float
        Local confinement index.
    count : int
        Number of surfaces contributing to the confinement estimate.

    Notes
    -----
    Each contributing surface is weighted by its effective height above
    the receiver, distance from the receiver, and orientation toward the
    receiver.
    """
    rx = np.asarray(rx, dtype=float)
    faces = np.asarray(faces_global, dtype=float)      
    centroids = faces.mean(axis=1)                     
    vec = rx - centroids                              
    distances = np.linalg.norm(vec, axis=1)           
    z_top = faces[:, :, 2].max(axis=1)                
    H_eff = np.maximum(z_top - rx[2], 0.0)           
    valid = distances > 1e-8
    score = np.zeros_like(distances)
    score[valid] = H_eff[valid] / distances[valid]
    mask = (H_eff > 0) & (score >= score_threshold)
    candidate_faces = faces[mask]
    candidate_vec = vec[mask]
    candidate_dist = distances[mask]
    candidate_H = H_eff[mask]
    confinement = 0.0
    count = 0
    for face, v, d, h in zip(candidate_faces,candidate_vec,candidate_dist,candidate_H):
        r = v / d
        normal = compute_face_normal(face)
        n_norm = np.linalg.norm(normal)
        if n_norm < 1e-12:
            continue
        normal /= n_norm
        orientation = abs(np.dot(normal, r))
        confinement += (h / d) * orientation
        count += 1
    if normalize and count > 0:
        confinement /= count
    return confinement, count





def plot_dxf_with_first_tx_rx(dxf_file, uav_coords, gnd_coords):
    """
    Visualize the DXF building footprints together with the first TX-RX pair.

    Parameters
    ----------
    dxf_file : path-like
        DXF building model.
    uav_coords : ndarray
        Transmitter coordinates.
    gnd_coords : ndarray
        Receiver coordinates.

    Notes
    -----
    This function is intended for qualitative inspection and debugging.
    It does not contribute to feature extraction.
    """
    doc = ezdxf.readfile(dxf_file)
    msp = doc.modelspace()
    types = Counter(entity.dxftype() for entity in msp)
    print(types)        
    fig, ax = plt.subplots(figsize=(10, 10))    
    for face in msp.query("3DFACE"):
        pts = []
        # DXF 3DFACE has four vertices
        for i in range(4):
            v = face.dxf.get(f"vtx{i}")
            pts.append((v.x, v.y))
        # Remove repeated vertex
        unique_pts = []
        for p in pts:
            if p not in unique_pts:
                unique_pts.append(p)
        poly = Polygon(unique_pts,closed=True,edgecolor="gray",facecolor="lightgray",linewidth=0.2)           
        ax.add_patch(poly)
    # Plot first TX
    tx = uav_coords[0]
    ax.scatter(tx[0], tx[1],c="red",s=120,marker="^",label="TX (UAV)")
    ax.text(tx[0],tx[1],"TX",color="red",fontsize=12)
    # Plot first RX
    rx = gnd_coords[0]
    ax.scatter(rx[0], rx[1],c="blue",s=80,marker="o",label="RX (Ground)")
    ax.text(rx[0],rx[1],"RX",color="blue",fontsize=12)
    # Line between TX and RX
    ax.plot([tx[0], rx[0]],[tx[1], rx[1]],"g--",linewidth=1)
    ax.set_aspect("equal")
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_title("DXF Buildings with First TX and RX")
    ax.legend()    
    ax.autoscale()
    plt.show()




In [6]:
def extract_all_features(dxf_file, output_file, uav_coords, gnd_coords, x_min=0, radial_bound=50 ):
    """
    Extract the complete geometric descriptor vector for all TX-RX pairs.

    Parameters
    ----------
    dxf_file : path-like
        DXF building model.
    output_file : path-like
        CSV file to which extracted descriptors are appended.
    uav_coords : ndarray, shape (N_tx, 3)
        Transmitter coordinates.
    gnd_coords : ndarray, shape (N_rx, 3)
        Receiver coordinates.
    x_min : float, default=0
        Minimum LOS coordinate used during candidate surface filtering.
    radial_bound : float, default=50
        Maximum perpendicular distance from the LOS for candidate surfaces.

    Returns
    -------
    pandas.DataFrame
        One feature record per TX-RX pair.

    Notes
    -----
    The pipeline consists of:
    
    1. DXF wall-face extraction.
    2. Link-geometry computation.
    3. LOS-oriented coordinate transformation.
    4. LOS candidate and intersection detection.
    5. Representative blocker extraction.
    6. Surrounding-surface reflection and blocking descriptors.
    7. Local confinement estimation.
    8. CSV feature export.

    The resulting columns correspond to the geometric descriptor groups
    defined in the methodology.
    """  
    faces = extract_faces_from_dxf(dxf_file)
    print("Number of wall faces:", len(faces))
    uav_coords = np.asarray(uav_coords)
    gnd_coords = np.asarray(gnd_coords)
    
    # Shape: (Ntx, Nrx, 3)
    diff = gnd_coords[None, :, :] - uav_coords[:, None, :]
    distance_3d_all = np.linalg.norm(diff, axis=2)
    distance_2d_all = np.linalg.norm(diff[:, :, :2], axis=2)
    height_difference_all = (uav_coords[:, None, 2] - gnd_coords[None, :, 2])
    elevation_angle_deg_all = np.degrees(np.arctan2(height_difference_all, distance_2d_all + 1e-12))
    records = []    
    for tx_idx, tx in enumerate(uav_coords):
        print("TX =", tx_idx)        
        for rx_idx, rx in enumerate(gnd_coords):
            # Precompute pairwise link-geometry quantities.
            distance_3d = distance_3d_all[tx_idx, rx_idx]            
            elevation_angle_deg = elevation_angle_deg_all[tx_idx, rx_idx]
            # Transform building faces into the LOS-oriented coordinate system.
            R = los_local_coordinates(tx, rx)
            faces_local = [(face - tx) @ R.T for face in faces]
            # Select surfaces near the LOS and identify exact LOS intersections.
            faces_local_filtered, face_indices, los_intersect, los_indices = filter_faces_along_los(faces_local, x_min, distance_3d, radial_bound)            
            los_faces_global = [faces[i] for i in face_indices]            
            los_faces_local = faces_local_filtered
            On_los_faces_global = [faces[i] for i in los_indices]
            On_los_faces_local = los_intersect
            # Extract the dominant LOS blocker.
            blocker = compute_representative_blocker(On_los_faces_global,On_los_faces_local,tx,rx)
            blocking_Candidate_number = blocker["blocking_Candidate_number"]
            blocking_score = blocker["blocking_score"]
            blocking_position = blocker["blocking_position"]
            blocking_alignment = blocker["blocking_alignment"]
            blocking_margin = blocker["blocking_margin"]
            blocking_Delta_height = blocker["blocking_Delta_height"] 
            # Extract receiver-side reflection, blocking, and occupancy descriptors.
            blocker_count,  sum_blocker = (compute_visible_blocking_features(los_faces_global,tx, rx))
            reflector_blocker = compute_reflector_blocker_features(los_faces_global,tx, rx)
            occupancy = reflector_blocker["occupancy"]
            reflection_potential = reflector_blocker["reflection_potential"]
            blocking_potential = reflector_blocker["blocking_potential"]
            effective_reflect_area = reflector_blocker["effective_reflect_area"]            
            log_reflection_ratio = reflector_blocker["log_reflection_ratio"]
            # Compute local spatial confinement around the receiver.
            local_confinement, normalize_local_confinement = compute_local_confinement_index(faces,rx,score_threshold=1.0)
            # Assemble one feature record for the current TX-RX pair.
            record = {
                "TX": tx_idx + 1,
                "RX": rx_idx + 1,

                "distance_3d": distance_3d,                
                "elevation_angle_deg": elevation_angle_deg,
                "tx_height": tx[2],
                "rx_height": rx[2],              

                "blocking_Candidate_number":blocking_Candidate_number,
                "blocking_score": blocking_score,
                "blocking_position": blocking_position,
                "blocking_alignment": blocking_alignment,
                "blocking_margin": blocking_margin,
                "blocking_Delta_height": blocking_Delta_height,

                "blocker_count": blocker_count,                
                "sum_blocker":sum_blocker,              

                "occupancy": occupancy,                
                "blocking_potential": blocking_potential,
                "reflection_potential": reflection_potential,
                "effective_reflect_area" : effective_reflect_area,               
                "log_reflection_ratio": log_reflection_ratio,
                 
                "local_confinement": local_confinement,
                "normalize_local_confinement":normalize_local_confinement,                                
            }
            # Append the record to the output CSV.
            df_record = pd.DataFrame([record])            
            if not os.path.exists(output_file):                
                df_record.to_csv(output_file, index=False)
            else:                
                df_record.to_csv(output_file, mode="a", header=False, index=False)                      
            records.append(record)            
    return pd.DataFrame(records)




## Run Experiments

The following cells execute the experiments for the proposed physics-motivated representation.


In [7]:
PLOT_DEBUG = False

for idx, city in cities.items():

    name = city["name"]
    uav_file = city["uav_file"]
    gnd_file = city["gnd_file"]
    dxf_file = city["dxf_file"]

    output_file = city["output_file"]    

    print(f"\nProcessing city {idx}: {name}")
    print("UAV:", uav_file)
    print("GND:", gnd_file)
    print("DXF:", dxf_file)

    # ---------------------------------------------------------
    # 1. Read UAV & GND coordinates
    # ---------------------------------------------------------
    uav_df, gnd_df, uav_coords, gnd_coords = read_tx_rx(uav_file,gnd_file)

    print("Loaded UAV:",uav_coords.shape,"Loaded GND:",gnd_coords.shape)
    # ---------------------------------------------------------
    # 2. Extract features
    # ---------------------------------------------------------    
    df = extract_all_features(dxf_file,output_file,uav_coords,gnd_coords,radial_bound=50)

    # ---------------------------------------------------------
    # 3. Plot
    # ---------------------------------------------------------
    if PLOT_DEBUG:
        plot_dxf_with_first_tx_rx(dxf_file,uav_coords,gnd_coords)

    print(f"Finished: {name}")



Processing city 1: Seoul
UAV: Inputs_DIR\Seoul_South Korea_UAV_Cartesian.csv
GND: Inputs_DIR\Seoul_South Korea_GND_Cartesian.csv
DXF: Inputs_DIR\Seoul_Buildings.dxf
Loaded UAV: (48, 3) Loaded GND: (625, 3)
Number of wall faces: 8510
TX = 0
Record 0- TX = 0 appended.
Finished: Seoul

Processing city 2: Boston
UAV: Inputs_DIR\BOSTON_UAV_Cartesian.csv
GND: Inputs_DIR\BOSTON_GND_Cartesian.csv
DXF: Inputs_DIR\Boston_Building.dxf
Loaded UAV: (30, 3) Loaded GND: (30, 3)
Number of wall faces: 16448
TX = 0
Record 0- TX = 0 appended.
Finished: Boston

Processing city 3: Pisa_East
UAV: Inputs_DIR\Pisa_East_UAV_Cartesian.csv
GND: Inputs_DIR\Pisa_East_GND_Cartesian.csv
DXF: Inputs_DIR\Pisa_East_Building.dxf
Loaded UAV: (30, 3) Loaded GND: (40, 3)
Number of wall faces: 17978
TX = 0
Record 0- TX = 0 appended.
Finished: Pisa_East

Processing city 4: Pisa_West
UAV: Inputs_DIR\Pisa_West_UAV_Cartesian.csv
GND: Inputs_DIR\Pisa_West_GND_Cartesian.csv
DXF: Inputs_DIR\Pisa_West_Building.dxf
Loaded UAV: (30,